# 01 — IST Exploratory Data Analysis

**Prerequisites:** download `IST_corrected.csv` from <https://datashare.ed.ac.uk/handle/10283/124> and place it at `data/raw/IST_corrected.csv`.

Goals of this notebook:
1. Read the raw file; check shape, dtypes, missingness.
2. Confirm the treatment-allocation columns (`RXASP`, `RXHEP`).
3. Inspect the outcome columns (`FDEAD`, `FDENNIS`, `OCCODE`, `ID14`).
4. Build the primary composite outcome and produce the first 2x2 cross-tab.
5. Persist the cleaned dataset to parquet.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data_loader, preprocess as pre

## 1. Load raw

In [ ]:
raw = data_loader.load_raw_ist()
print(raw.shape)
raw.head()

In [ ]:
summary = data_loader.quick_summary(raw)
summary.sort_values('pct_missing', ascending=False).head(30)

## 2. Treatment columns

In [ ]:
print('Aspirin allocation:'); print(raw['RXASP'].value_counts(dropna=False))
print('\nHeparin allocation:'); print(raw['RXHEP'].value_counts(dropna=False))

## 3. Outcome columns

In [ ]:
for col in ['FDEAD', 'FDENNIS', 'ID14', 'OCCODE']:
    if col in raw.columns:
        print(col, '->', raw[col].value_counts(dropna=False).to_dict())

## 4. Run the preprocessing pipeline

In [ ]:
clean = pre.preprocess(raw)
print(clean.shape)
clean[['treatment', 'outcome', 'AGE', 'SEX', 'RSBP', 'RCONSC']].head()

In [ ]:
# Marginal event rates by arm — the 'average treatment effect'
rates = clean.groupby('treatment')['outcome'].agg(['count', 'mean']).round(4)
rates.columns = ['n', 'event_rate']
print(rates)
print(f'\nMarginal ARR: {rates.loc[0, "event_rate"] - rates.loc[1, "event_rate"]:.4f}')

## 5. Persist cleaned data

In [ ]:
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
clean.to_parquet(config.IST_CLEAN_PATH, index=False)
print('Wrote', config.IST_CLEAN_PATH)